In [4]:
# ============================================================
# CREDRESOLVE — COLLECTIONS RECOVERY ANALYTICS
# 03 — GOLDEN DATASET CONSTRUCTION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
STAGING_DIR = PROJECT_ROOT / "data" / "staging"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
GOLDEN_DIR = PROJECT_ROOT / "data" / "golden"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"

for directory in [
    STAGING_DIR,
    PROCESSED_DIR,
    GOLDEN_DIR,
    OUTPUT_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 90)
print("CREDRESOLVE — GOLDEN DATASET CONSTRUCTION")
print("=" * 90)


# ------------------------------------------------------------
# 2. LOAD RAW DATA
# ------------------------------------------------------------

csv_files = sorted(RAW_DIR.glob("*.csv"))

datasets = {
    file.stem: pd.read_csv(file)
    for file in csv_files
}

print(f"Raw datasets loaded: {len(datasets)}")


# ------------------------------------------------------------
# 3. GENERIC GOLDEN COPY
# ------------------------------------------------------------

golden_tables = {}
cleaning_log = []

for name, df in datasets.items():

    original_rows = len(df)

    golden_df = df.copy()

    # Audit source row
    golden_df["_source_row_number"] = np.arange(
        1,
        len(golden_df) + 1
    )

    # Remove only exact duplicate rows
    duplicate_mask = golden_df.duplicated(
        keep="first"
    )

    duplicate_count = int(
        duplicate_mask.sum()
    )

    golden_df = golden_df[
        ~duplicate_mask
    ].copy()

    # Trim whitespace from text columns
    for column in golden_df.select_dtypes(
        include=["object"]
    ).columns:

        golden_df[column] = (
            golden_df[column]
            .astype("string")
            .str.strip()
        )

    final_rows = len(golden_df)

    cleaning_log.append({
        "dataset": name,
        "original_rows": original_rows,
        "exact_duplicate_rows_removed": duplicate_count,
        "golden_rows": final_rows,
        "rows_retained_pct": round(
            final_rows / original_rows * 100,
            2
        ) if original_rows else 0,
        "raw_data_modified": False
    })

    golden_tables[name] = golden_df


# ============================================================
# 4. TIMESTAMP STANDARDIZATION
# ============================================================

# PAYMENTS
if "payments" in golden_tables:

    df = golden_tables["payments"]

    df["event_at_standardized"] = pd.to_datetime(
        df["event_at"],
        errors="coerce"
    )

    df["event_date"] = (
        df["event_at_standardized"].dt.date
    )

    df["event_month"] = (
        df["event_at_standardized"]
        .dt.to_period("M")
        .astype("string")
    )

    df["timestamp_parse_failed"] = (
        df["event_at"].notna()
        &
        df["event_at_standardized"].isna()
    )

    golden_tables["payments"] = df


# CALLS
if "calls" in golden_tables:

    df = golden_tables["calls"]

    df["event_at_standardized"] = pd.to_datetime(
        df["event_at"],
        errors="coerce"
    )

    df["event_date"] = (
        df["event_at_standardized"].dt.date
    )

    df["event_month"] = (
        df["event_at_standardized"]
        .dt.to_period("M")
        .astype("string")
    )

    df["timestamp_parse_failed"] = (
        df["event_at"].notna()
        &
        df["event_at_standardized"].isna()
    )

    golden_tables["calls"] = df


# CALL ATTEMPTS
if "call_attempts" in golden_tables:

    df = golden_tables["call_attempts"]

    df["event_at_standardized"] = pd.to_datetime(
        df["event_at"],
        errors="coerce"
    )

    df["event_date"] = (
        df["event_at_standardized"].dt.date
    )

    df["event_month"] = (
        df["event_at_standardized"]
        .dt.to_period("M")
        .astype("string")
    )

    df["timestamp_parse_failed"] = (
        df["event_at"].notna()
        &
        df["event_at_standardized"].isna()
    )

    golden_tables["call_attempts"] = df


# CALL DISPOSITIONS
if "call_dispositions" in golden_tables:

    df = golden_tables["call_dispositions"]

    df["event_at_standardized"] = pd.to_datetime(
        df["event_at"],
        errors="coerce"
    )

    df["event_date"] = (
        df["event_at_standardized"].dt.date
    )

    df["event_month"] = (
        df["event_at_standardized"]
        .dt.to_period("M")
        .astype("string")
    )

    df["timestamp_parse_failed"] = (
        df["event_at"].notna()
        &
        df["event_at_standardized"].isna()
    )

    golden_tables["call_dispositions"] = df


# ACCOUNTS
if "accounts" in golden_tables:

    df = golden_tables["accounts"]

    df["opened_at_standardized"] = pd.to_datetime(
        df["opened_at"],
        errors="coerce"
    )

    df["opened_month"] = (
        df["opened_at_standardized"]
        .dt.to_period("M")
        .astype("string")
    )

    df["timestamp_parse_failed"] = (
        df["opened_at"].notna()
        &
        df["opened_at_standardized"].isna()
    )

    golden_tables["accounts"] = df


# DAILY TARGETING
if "daily_targeting" in golden_tables:

    df = golden_tables["daily_targeting"]

    df["target_date_standardized"] = pd.to_datetime(
        df["target_date"],
        errors="coerce"
    )

    df["target_month"] = (
        df["target_date_standardized"]
        .dt.to_period("M")
        .astype("string")
    )

    df["timestamp_parse_failed"] = (
        df["target_date"].notna()
        &
        df["target_date_standardized"].isna()
    )

    golden_tables["daily_targeting"] = df


# AGENT SESSIONS
if "agent_sessions" in golden_tables:

    df = golden_tables["agent_sessions"]

    df["login_at_standardized"] = pd.to_datetime(
        df["login_at"],
        errors="coerce"
    )

    df["logout_at_standardized"] = pd.to_datetime(
        df["logout_at"],
        errors="coerce"
    )

    df["session_duration_minutes"] = (
        (
            df["logout_at_standardized"]
            -
            df["login_at_standardized"]
        )
        .dt.total_seconds()
        / 60
    )

    df["invalid_session_time"] = (
        df["logout_at_standardized"]
        <
        df["login_at_standardized"]
    )

    golden_tables["agent_sessions"] = df


# ============================================================
# 5. PAYMENT FORENSIC FLAGS
# ============================================================

if "payments" in golden_tables:

    df = golden_tables["payments"]

    # Duplicate payment ID
    id_counts = (
        df.groupby("payment_id")
        .size()
        .rename("payment_id_occurrences")
    )

    df = df.merge(
        id_counts,
        left_on="payment_id",
        right_index=True,
        how="left"
    )

    df["duplicate_payment_id_flag"] = (
        df["payment_id_occurrences"] > 1
    )

    # Duplicate payment reference
    reference_counts = (
        df.dropna(subset=["payment_reference"])
        .groupby("payment_reference")
        .size()
        .rename(
            "payment_reference_occurrences"
        )
    )

    df = df.merge(
        reference_counts,
        left_on="payment_reference",
        right_index=True,
        how="left"
    )

    df["duplicate_payment_reference_flag"] = (
        df["payment_reference_occurrences"] > 1
    )

    golden_tables["payments"] = df


# ============================================================
# 6. ACCOUNT / BORROWER CONSISTENCY
# ============================================================

if (
    "accounts" in golden_tables
    and "payments" in golden_tables
):

    account_map = (
        golden_tables["accounts"][
            [
                "account_id",
                "borrower_id"
            ]
        ]
        .drop_duplicates("account_id")
        .rename(
            columns={
                "borrower_id":
                "account_borrower_id"
            }
        )
    )

    df = golden_tables["payments"].merge(
        account_map,
        on="account_id",
        how="left"
    )

    df["borrower_account_mismatch_flag"] = (
        df["borrower_id"]
        !=
        df["account_borrower_id"]
    )

    df["borrower_account_mismatch_flag"] = (
        df["borrower_account_mismatch_flag"]
        &
        df["account_borrower_id"].notna()
    )

    golden_tables["payments"] = df


# ============================================================
# 7. CALL / ACCOUNT BORROWER CONSISTENCY
# ============================================================

if (
    "accounts" in golden_tables
    and "calls" in golden_tables
):

    account_map = (
        golden_tables["accounts"][
            [
                "account_id",
                "borrower_id"
            ]
        ]
        .drop_duplicates("account_id")
        .rename(
            columns={
                "borrower_id":
                "account_borrower_id"
            }
        )
    )

    df = golden_tables["calls"].merge(
        account_map,
        on="account_id",
        how="left"
    )

    df["borrower_account_mismatch_flag"] = (
        df["borrower_id"]
        !=
        df["account_borrower_id"]
    )

    df["borrower_account_mismatch_flag"] = (
        df["borrower_account_mismatch_flag"]
        &
        df["account_borrower_id"].notna()
    )

    golden_tables["calls"] = df


# ============================================================
# 8. CORRECT TIMEZONE VALIDATION
# ============================================================
#
# IMPORTANT:
# Do NOT compare call timezone to vendor timezone.
#
# vendor_telephony.timezone = vendor operating timezone
# accounts.timezone         = account/customer timezone
# calls.timezone             = timezone recorded for call
#
# We compare call timezone with account timezone as an
# investigation signal only.
# ============================================================

if (
    "calls" in golden_tables
    and "accounts" in golden_tables
):

    account_timezone_map = (
        golden_tables["accounts"][
            [
                "account_id",
                "timezone"
            ]
        ]
        .drop_duplicates("account_id")
        .rename(
            columns={
                "timezone":
                "account_timezone"
            }
        )
    )

    df = golden_tables["calls"].merge(
        account_timezone_map,
        on="account_id",
        how="left"
    )

    df["account_timezone_comparison_flag"] = (
        df["timezone"]
        !=
        df["account_timezone"]
    )

    df["account_timezone_comparison_flag"] = (
        df["account_timezone_comparison_flag"]
        &
        df["account_timezone"].notna()
        &
        df["timezone"].notna()
    )

    golden_tables["calls"] = df


# ============================================================
# 9. AGENT VALIDATION
# ============================================================

if (
    "calls" in golden_tables
    and "agents" in golden_tables
):

    valid_agents = set(
        golden_tables["agents"][
            "agent_id"
        ].dropna()
    )

    df = golden_tables["calls"]

    df["unknown_agent_flag"] = (
        df["agent_id"].notna()
        &
        ~df["agent_id"].isin(valid_agents)
    )

    golden_tables["calls"] = df


# ============================================================
# 10. CAMPAIGN VALIDATION
# ============================================================

if (
    "calls" in golden_tables
    and "campaigns" in golden_tables
):

    valid_campaigns = set(
        golden_tables["campaigns"][
            "campaign_id"
        ].dropna()
    )

    df = golden_tables["calls"]

    df["unknown_campaign_flag"] = (
        df["campaign_id"].notna()
        &
        ~df["campaign_id"].isin(
            valid_campaigns
        )
    )

    golden_tables["calls"] = df


if (
    "daily_targeting" in golden_tables
    and "campaigns" in golden_tables
):

    valid_campaigns = set(
        golden_tables["campaigns"][
            "campaign_id"
        ].dropna()
    )

    df = golden_tables[
        "daily_targeting"
    ]

    df["unknown_campaign_flag"] = (
        df["campaign_id"].notna()
        &
        ~df["campaign_id"].isin(
            valid_campaigns
        )
    )

    golden_tables[
        "daily_targeting"
    ] = df


# ============================================================
# 11. SAVE GOLDEN TABLES
# ============================================================

for name, df in golden_tables.items():

    output_file = (
        GOLDEN_DIR /
        f"{name}_golden.csv"
    )

    df.to_csv(
        output_file,
        index=False
    )


# ============================================================
# 12. CLEANING AUDIT LOG
# ============================================================

cleaning_log_df = pd.DataFrame(
    cleaning_log
)

cleaning_log_df.to_csv(
    OUTPUT_DIR /
    "golden_dataset_cleaning_log.csv",
    index=False
)


# ============================================================
# 13. GOLDEN DATASET SUMMARY
# ============================================================

golden_summary = []

for name, df in golden_tables.items():

    golden_summary.append({

        "dataset": name,

        "golden_rows": len(df),

        "columns": len(df.columns),

        "missing_cells": int(
            df.isna().sum().sum()
        ),

        "duplicate_rows_remaining": int(
            df.duplicated().sum()
        )
    })

golden_summary_df = pd.DataFrame(
    golden_summary
).sort_values("dataset")

print("\n" + "=" * 90)
print("GOLDEN DATASET SUMMARY")
print("=" * 90)

display(
    golden_summary_df
)

golden_summary_df.to_csv(
    OUTPUT_DIR /
    "golden_dataset_summary.csv",
    index=False
)


# ============================================================
# 14. INVESTIGATION FLAG SUMMARY
# ============================================================

flag_summary = []

for name, df in golden_tables.items():

    flag_columns = [
        column
        for column in df.columns
        if column.endswith("_flag")
    ]

    for column in flag_columns:

        flag_summary.append({

            "dataset": name,

            "flag": column,

            "flagged_rows": int(
                df[column]
                .fillna(False)
                .sum()
            )
        })

flag_summary_df = pd.DataFrame(
    flag_summary
)

print("\n" + "=" * 90)
print("GOLDEN DATASET INVESTIGATION FLAGS")
print("=" * 90)

display(
    flag_summary_df
)

flag_summary_df.to_csv(
    OUTPUT_DIR /
    "golden_dataset_flag_summary.csv",
    index=False
)


# ============================================================
# 15. FINAL STATUS
# ============================================================

print("\n" + "=" * 90)
print("GOLDEN DATASET CONSTRUCTION COMPLETE")
print("=" * 90)

print(
    f"Golden tables created: {len(golden_tables)}"
)

print(
    f"Golden data location: {GOLDEN_DIR}"
)

print(
    f"Audit outputs location: {OUTPUT_DIR}"
)

print(
    "Raw source files were NOT modified."
)

print(
    "Exact duplicate rows were removed only "
    "from Golden copies."
)

print(
    "Payment duplicate IDs/references were flagged, "
    "not automatically deleted."
)

print(
    "Timezone differences are investigation flags, "
    "not automatic data errors."
)

print(
    "Original source timestamps and identifiers were retained."
)

CREDRESOLVE — GOLDEN DATASET CONSTRUCTION
Raw datasets loaded: 18

GOLDEN DATASET SUMMARY


,dataset,golden_rows,columns,missing_cells,duplicate_rows_remaining
0,account_status_history,60000,9,0,0
1,accounts,30000,15,455,0
2,agent_sessions,15000,12,0,0
3,agents,30000,9,0,0
4,borrowers,30600,9,1509,0
5,call_attempts,120000,14,2400,0
6,call_dispositions,35000,13,0,0
7,calls,91350,22,3230,0
8,campaigns,120,8,0,0
9,complaints,8000,10,0,0



GOLDEN DATASET INVESTIGATION FLAGS


,dataset,flag,flagged_rows
0,calls,borrower_account_mismatch_flag,89939
1,calls,account_timezone_comparison_flag,60887
2,calls,unknown_agent_flag,0
3,calls,unknown_campaign_flag,0
4,daily_targeting,unknown_campaign_flag,0
5,payments,duplicate_payment_id_flag,1000
6,payments,duplicate_payment_reference_flag,8042
7,payments,borrower_account_mismatch_flag,25113



GOLDEN DATASET CONSTRUCTION COMPLETE
Golden tables created: 18
Golden data location: c:\Users\DELL\Documents\CredResolve Collections Recovery Analytics\data\golden
Audit outputs location: c:\Users\DELL\Documents\CredResolve Collections Recovery Analytics\outputs\tables
Raw source files were NOT modified.
Exact duplicate rows were removed only from Golden copies.
Payment duplicate IDs/references were flagged, not automatically deleted.
Timezone differences are investigation flags, not automatic data errors.
Original source timestamps and identifiers were retained.
